# 10. Penalty coefficient(lambda) 민감도 실험

## 왜 이 실험을 하는가

본 실험(notebook 04, 05)에서는 lambda를 시작 전에 고정하고 결과를 보고 조정하지 않았습니다. 그 결과 QA는 24회 중 1회만 feasible solution을 냈고, QUBO 계수 범위가 10^9에 달했습니다.

이 notebook은 **본 실험의 일부가 아니라 그 결과를 진단하기 위한 별도 실험**입니다. lambda를 의도적으로 바꿔가며 두 가설을 구분합니다.

| 가설 | 내용 | 예측 |
|---|---|---|
| A. precision | lambda가 커서 계수 범위가 넓어지고, objective 정보가 하드웨어의 아날로그 정밀도 아래로 묻힌다 | lambda를 낮추면 **QA만** 개선 |
| B. landscape | lambda가 커서 penalty 지형이 험해지고, solver가 지형 자체를 탐색하지 못한다 | lambda를 낮추면 **SA와 QA가 함께** 개선 |

SA는 소프트웨어라 아날로그 정밀도 제약이 없습니다. 따라서 **SA를 classical control로 두면 두 가설이 구분됩니다.** 이것이 이 실험 설계의 핵심입니다.

## 대상

**SS 8x8**을 주 대상으로 합니다. 이유는 세 가지입니다.

- embedding에 성공했습니다. 실패하는 조합에서는 lambda를 바꿔도 관측할 것이 없습니다.
- 그럼에도 QA feasible 비율이 0%였습니다. 개선 여지가 관측 가능합니다.
- QUBO 변수가 117개로 QPU 시간을 과하게 쓰지 않습니다.

여유가 있으면 SS 4x4(QA가 유일하게 성공한 경우)와 MS 4x4(embedding은 됐으나 feasible 0)를 추가로 돌려 비교하십시오.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


설정 로드 완료


## lambda grid 설계

lambda의 이론적 하한은 U_obj가 아니라 **Z_ub**입니다.

infeasible solution의 energy는 objective가 0 이상이므로 최소 lambda입니다. 한편 optimal solution의 energy Z*는 알려진 feasible solution의 objective Z_ub 이하입니다. 따라서

$$\lambda > Z_{ub} \implies \text{ground state가 feasible}$$

이 성립합니다. 현재 설정은 `lambda = 1.1 x U_obj`인데, SS 8x8에서 이 값은 Z_ub의 약 4.56배입니다. 즉 필요한 것보다 4.5배 큽니다.

그래서 grid를 `m = lambda / Z_ub`로 두고 **m = 1이 이론적 경계**가 되도록 했습니다.

- `m < 1` : 이론적 보장이 깨지는 영역. **진단 목적으로만** 사용하고 본 실험 결과로는 보고하지 않습니다.
- `m = 1` : 이론적 경계
- `m = 4.56` : 현재 설정


In [2]:
from src.data_generator import CFLPInstance
from src.persistence import load_solution, save_table
from src.decoder import encode_solution
from src.qubo_builder import build_qubo
from src import lambda_experiment as LE

TARGET_INSTANCE = "8x8"
TARGET_FORMULATION = "SS"

instance = CFLPInstance.load(DATA_DIR / f"{TARGET_INSTANCE}.json")
gurobi = pd.read_csv(RAW_DIR / "gurobi_results.csv")
model_name = "SS" if TARGET_FORMULATION == "SS" else "MS"
reference_objective = float(
    gurobi[(gurobi["instance"] == TARGET_INSTANCE)
           & (gurobi["gurobi_model"] == model_name)]["objective"].iloc[0]
)
reference_solution = load_solution(
    SOLUTION_DIR, TARGET_INSTANCE, f"gurobi_{model_name.replace('-', '_')}"
)
base_model = build_qubo(instance, TARGET_FORMULATION, float(config["penalty"]["margin"]))
reference_sample = encode_solution(base_model, instance, reference_solution)

current_multiplier = base_model.penalty.value / reference_objective
print(f"Z_ub (Gurobi optimum) = {reference_objective:.2f}")
print(f"현재 lambda           = {base_model.penalty.value:.2f}")
print(f"현재 m = lambda/Z_ub  = {current_multiplier:.2f}")
print(f"grid                  = {LE.DEFAULT_MULTIPLIERS}")

Z_ub (Gurobi optimum) = 3952.52
현재 lambda           = 18024.59
현재 m = lambda/Z_ub  = 4.56
grid                  = (0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0, 7.5, 10.0)


## SA sweep (classical control)

SA는 아날로그 정밀도 제약이 없으므로, SA가 lambda에 어떻게 반응하는지가 **가설 B의 크기를 직접 알려줍니다.**

SA 파라미터는 본 실험과 동일하게 유지합니다. 바꾸는 것은 lambda 하나뿐입니다.

In [3]:
sa_points = LE.run_sa_sweep(
    instance=instance,
    formulation=TARGET_FORMULATION,
    reference_objective=reference_objective,
    reference_sample=reference_sample,
    sa_config=config["sa"],
    tolerance=float(config["feasibility"]["tolerance"]),
)
sa_frame = pd.DataFrame([point.to_record() for point in sa_points])
sa_frame.insert(0, "instance", TARGET_INSTANCE)
sa_frame.insert(1, "formulation", TARGET_FORMULATION)
sa_frame[[
    "multiplier", "lambda_value", "qubo_range",
    "scaled_objective_coefficient", "feasible_fraction",
    "true_gap_percent", "ground_state_is_feasible",
]]

,multiplier,lambda_value,qubo_range,scaled_objective_coefficient,feasible_fraction,true_gap_percent,ground_state_is_feasible
0,0.10,395.251530,1.581286e+07,7.230743e-06,0.001,37.761627,True
1,0.25,988.128825,3.953032e+07,2.892297e-06,0.000,NaN,True
2,0.50,1976.257650,7.905943e+07,1.446149e-06,0.001,70.647000,True
3,0.75,2964.386475,1.185885e+08,9.640991e-07,0.000,NaN,True
4,1.00,3952.515300,1.581176e+08,7.230743e-07,0.002,67.654443,True
5,1.25,4940.644125,1.976467e+08,5.784595e-07,0.001,59.697854,True
6,1.50,5928.772950,2.371758e+08,4.820495e-07,0.000,NaN,True
7,2.00,7905.030600,3.162341e+08,3.615372e-07,0.001,45.155800,True
8,3.00,11857.545900,4.743505e+08,2.410248e-07,0.001,69.206497,True
9,5.00,19762.576500,7.905833e+08,1.446149e-07,0.002,44.002023,True


## 하드웨어 정밀도와의 비교

`scaled_objective_coefficient`는 D-Wave의 auto_scale 이후 objective 계수가 얼마나 작아지는지를 나타냅니다. D-Wave는 모든 계수를 max|J|로 나누어 하드웨어 범위에 맞추기 때문입니다.

ICE(integrated control error)에 의한 노이즈는 정규화 단위로 대략 0.01~0.03 수준으로 알려져 있습니다. objective 계수가 이보다 작으면 **하드웨어가 objective를 구분할 수 없다**는 뜻입니다.

In [4]:
ICE_NOISE = 0.02   # 정규화 단위. 정확한 값이 아니라 자릿수 비교용 기준.
view = sa_frame[["multiplier", "lambda_value", "scaled_objective_coefficient"]].copy()
view["noise_ratio"] = view["scaled_objective_coefficient"] / ICE_NOISE
view["objective_visible"] = view["noise_ratio"] > 1.0
print("objective 계수가 ICE 노이즈보다 큰 lambda가 하나라도 있는가:",
      bool(view["objective_visible"].any()))
view

objective 계수가 ICE 노이즈보다 큰 lambda가 하나라도 있는가: False


,multiplier,lambda_value,scaled_objective_coefficient,noise_ratio,objective_visible
0,0.10,241.68364,0.000211,0.010546,False
1,0.25,604.20910,0.000084,0.004218,False
2,0.50,1208.41820,0.000042,0.002109,False
3,0.75,1812.62730,0.000028,0.001406,False
4,1.00,2416.83640,0.000021,0.001055,False
5,1.25,3021.04550,0.000017,0.000844,False
6,1.50,3625.25460,0.000014,0.000703,False
7,2.00,4833.67280,0.000011,0.000527,False
8,3.00,7250.50920,0.000007,0.000352,False
9,5.00,12084.18200,0.000004,0.000211,False


## QA sweep

**중요**: QUBO의 그래프 구조는 lambda와 무관합니다. lambda는 계수의 크기만 바꿀 뿐 어떤 변수쌍이 연결되는지는 바꾸지 않습니다. 따라서 embedding을 **한 번만 계산해 모든 lambda에 재사용**합니다.

이렇게 해야 관측된 차이가 embedding 운이 아니라 lambda 때문임을 보장할 수 있습니다.

chain strength는 `alpha x max|J|` 규칙이므로 lambda에 따라 자동으로 함께 움직입니다. 즉 chain의 상대적 강도는 일정하게 유지됩니다.

토큰이 없으면 이 셀은 건너뛰고 SA 결과만으로 진행합니다.

In [5]:
qa_frame = pd.DataFrame()

qa_points = LE.run_qa_sweep(
    instance=instance,
    formulation=TARGET_FORMULATION,
    reference_objective=reference_objective,
    reference_sample=reference_sample,
    qa_config=config["qa"],
    embedding_config=config["embedding"],
    tolerance=float(config["feasibility"]["tolerance"]),
)

qa_frame = pd.DataFrame([point.to_record() for point in qa_points])
qa_frame.insert(0, "instance", TARGET_INSTANCE)
qa_frame.insert(1, "formulation", TARGET_FORMULATION)

display(qa_frame[[
    "multiplier",
    "lambda_value",
    "feasible_fraction",
    "true_gap_percent",
    "qa_chain_break_fraction",
]])

,multiplier,lambda_value,feasible_fraction,true_gap_percent,qa_chain_break_fraction
0,0.10,241.68364,0.007,22.920923,0.019932
1,0.25,604.20910,0.001,27.659638,0.021591
2,0.50,1208.41820,0.006,7.695453,0.019250
3,0.75,1812.62730,0.007,20.366985,0.021545
4,1.00,2416.83640,0.004,0.402799,0.018659
5,1.25,3021.04550,0.002,29.672600,0.020273
6,1.50,3625.25460,0.007,0.402799,0.020091
7,2.00,4833.67280,0.004,27.057942,0.020977
8,3.00,7250.50920,0.006,0.402799,0.018750
9,5.00,12084.18200,0.005,0.402799,0.020818


In [6]:
lambda_results = pd.concat([sa_frame, qa_frame], ignore_index=True)
save_table(lambda_results, PROCESSED_DIR, "lambda_sweep.csv")
print("저장:", PROCESSED_DIR / "lambda_sweep.csv", f"({len(lambda_results)} 행)")

저장: C:\Users\User\Desktop\KMJ\Study\Quantum\cflp_formulation\results\processed\lambda_sweep.csv (24 행)


## 가설 판정

lambda가 낮은 구간(m <= 1)과 높은 구간(m >= 3)의 feasible 비율을 비교합니다.

In [7]:
for line in LE.interpret(lambda_results):
    print(line)

SA: lambda 낮은 구간 feasible 0.013 vs 높은 구간 0.010 (차이 +0.003)
QA: lambda 낮은 구간 feasible 0.005 vs 높은 구간 0.006 (차이 -0.001)
=> 둘 다 반응 없음. lambda 외의 요인을 찾아야 합니다.
